# Project 3 — Enterprise Human Approval Workflow

A stateful, risk-aware, **human-in-the-loop** AI workflow orchestrated with LangGraph.

This notebook generates the complete `03-human-approval-workflow/` project from scratch,
installs dependencies, configures Gemini, and then **actually runs** every branch of the
workflow:

1. A low-risk request that executes automatically.
2. A high-risk request that genuinely **pauses** via `interrupt()`, then resumes with
   `Command(resume="approve")`.
3. A second high-risk request that pauses and resumes with `Command(resume="reject")`.
4. The full pytest suite (mocked LLM, no API key required for tests).

Run the cells top to bottom. You will need a Gemini API key from
[Google AI Studio](https://aistudio.google.com/apikey) stored in Colab's Secrets
manager under the name `GOOGLE_API_KEY` (see Cell 17).


In [ ]:
# Cell 2 — Install dependencies
!pip install -q "langgraph>=1.2.0,<2.0.0" "langchain-core>=1.2.0,<2.0.0" \
    "langchain-google-genai>=4.3.0,<5.0.0" "pydantic>=2.7.0,<3.0.0" \
    "python-dotenv>=1.0.0,<2.0.0" "pytest>=8.0.0,<9.0.0"
print("Dependencies installed.")


In [ ]:
# Cell 3 — Create project folders
import os

PROJECT_DIR = "03-human-approval-workflow"

for sub in ["", "tests", "examples"]:
    os.makedirs(os.path.join(PROJECT_DIR, sub), exist_ok=True)

print(f"Created {PROJECT_DIR}/ with tests/ and examples/ subfolders.")


## Generate project files

Each cell below writes one real source file into `03-human-approval-workflow/` using `%%writefile`. Nothing is pseudo-code — every file here is the exact, tested implementation.

In [ ]:
%%writefile 03-human-approval-workflow/requirements.txt
langgraph>=1.2.0,<2.0.0
langchain-core>=1.2.0,<2.0.0
langchain-google-genai>=4.3.0,<5.0.0
pydantic>=2.7.0,<3.0.0
python-dotenv>=1.0.0,<2.0.0
pytest>=8.0.0,<9.0.0


In [ ]:
%%writefile 03-human-approval-workflow/.gitignore
.env
.env.*
!.env.example

__pycache__/
*.pyc

.ipynb_checkpoints/

.pytest_cache/

*.egg-info/
.DS_Store


In [ ]:
%%writefile 03-human-approval-workflow/.env.example
# Copy this file to .env and fill in your own values.
# NEVER commit your real .env file.

# Google AI Studio API key used for Gemini risk-assessment calls.
GOOGLE_API_KEY=your-google-api-key-here

# Gemini model used for structured risk assessment.
GEMINI_MODEL=gemini-2.0-flash

# Risk score threshold (0-100). Scores >= this value require human approval.
HIGH_RISK_THRESHOLD=40


In [ ]:
%%writefile 03-human-approval-workflow/config.py
"""
Configuration for the Enterprise Human Approval Workflow.

All configuration is read from environment variables. Nothing sensitive
is hardcoded here. In Colab, credentials should be injected via
`google.colab.userdata` before this module is imported (see the README
and the generated notebook).
"""

from __future__ import annotations

import os
from dataclasses import dataclass

from dotenv import load_dotenv

# Loads a local .env file if present. In Colab, environment variables are
# instead set directly via google.colab.userdata (see README / notebook).
load_dotenv()


@dataclass(frozen=True)
class Settings:
    google_api_key: str
    gemini_model: str
    high_risk_threshold: int


def _get_int(name: str, default: int) -> int:
    raw = os.environ.get(name)
    if raw is None or raw.strip() == "":
        return default
    try:
        return int(raw)
    except ValueError:
        return default


def load_settings() -> Settings:
    """Load settings from the current process environment.

    Called lazily (not at import time) so that test suites and CLI tools
    that never touch the LLM do not require GOOGLE_API_KEY to be set.
    """
    return Settings(
        google_api_key=os.environ.get("GOOGLE_API_KEY", ""),
        gemini_model=os.environ.get("GEMINI_MODEL", "gemini-2.0-flash"),
        high_risk_threshold=_get_int("HIGH_RISK_THRESHOLD", 40),
    )


In [ ]:
%%writefile 03-human-approval-workflow/state.py
"""
Graph state definition for the Enterprise Human Approval Workflow.

The state is the single source of truth that flows through every node.
It represents the complete lifecycle of a request:

    REQUEST -> ANALYSIS -> RISK -> APPROVAL DECISION -> HUMAN DECISION -> EXECUTION

Nothing about routing, execution, or approval lives outside this state;
the graph reads and writes it at every step, which is what makes the
workflow inspectable, auditable, and resumable.
"""

from __future__ import annotations

from typing import Literal, TypedDict


class ApprovalState(TypedDict, total=False):
    # --- Request identity -------------------------------------------------
    request_id: str
    requester: str
    department: str

    # --- Request payload ----------------------------------------------------
    action: str
    amount: float
    reason: str

    # --- Risk assessment (produced by analyze_request) ----------------------
    risk_score: int
    risk_level: Literal["LOW", "HIGH"]
    risk_reason: str
    risk_factors: list[str]

    # --- Policy decision (produced deterministically from risk_score) -------
    approval_required: bool
    approval_status: Literal["not_required", "pending", "approved", "rejected"]

    # --- Human decision (produced by human_approval / interrupt) ------------
    human_decision: str

    # --- Execution outcome ----------------------------------------------------
    execution_status: Literal["not_started", "executed", "rejected", "failed"]
    execution_result: str

    # --- Governance / audit ---------------------------------------------------
    errors: list[str]
    metadata: dict


def new_state(
    requester: str,
    department: str,
    action: str,
    amount: float,
    reason: str,
) -> ApprovalState:
    """Build a fresh, minimally-populated state for a new request.

    Everything else (request_id, risk fields, approval fields, execution
    fields) is filled in by the graph nodes as the workflow progresses.
    """
    return ApprovalState(
        requester=requester,
        department=department,
        action=action,
        amount=amount,
        reason=reason,
        errors=[],
        metadata={},
        approval_status="not_required",
        execution_status="not_started",
    )


In [ ]:
%%writefile 03-human-approval-workflow/prompts.py
"""
Prompt templates for the risk-assessment LLM call.

Important architectural note (see README section "Policy vs LLM"):
the model is asked only to *assess* risk and explain its reasoning.
It is never asked to decide whether to execute anything. That decision
is made afterwards by a deterministic policy layer (a configurable
score threshold) in nodes.py / graph.py.
"""

RISK_ASSESSMENT_PROMPT = """You are a risk-assessment analyst supporting an enterprise \
approval workflow. You will be shown a single business request. Your job is \
ONLY to assess its risk — you do not decide whether it should be approved, \
executed, or rejected. That decision belongs to a separate policy system.

Evaluate the request across these dimensions:
- Financial exposure (how much money / value is at stake)
- Operational impact (how much of the business this could disrupt)
- Access sensitivity (does this touch production systems, credentials, or PII)
- Potential security implications
- Reversibility (how hard would this be to undo if it were wrong)
- Business importance / urgency

Base your assessment strictly on the information provided. Do not invent \
facts about the requester, the company, or systems that were not mentioned. \
If information is missing, treat that as increased uncertainty rather than \
filling in assumptions.

Return a risk score from 0 (trivial, no concern) to 100 (severe, high-stakes, \
irreversible). Also return a short list of the specific risk factors that \
drove your score, and a concise, plain-language explanation of your reasoning.

Request details:
- Requester: {requester}
- Department: {department}
- Action: {action}
- Amount: {amount}
- Reason: {reason}
"""


In [ ]:
%%writefile 03-human-approval-workflow/validators.py
"""
Input validation for incoming approval requests.

This module never calls the LLM. Validation is pure, deterministic
Python — if a request is malformed, we reject it before spending a
single token on it.
"""

from __future__ import annotations

from state import ApprovalState
from utils import generate_request_id


def validate_fields(
    requester: str,
    department: str,
    action: str,
    amount: float,
    reason: str,
) -> list[str]:
    """Return a list of human-readable validation errors (empty if valid)."""
    errors: list[str] = []

    if not requester or not str(requester).strip():
        errors.append("requester must not be empty")

    if not department or not str(department).strip():
        errors.append("department must not be empty")

    if not action or not str(action).strip():
        errors.append("action must not be empty")

    if not reason or not str(reason).strip():
        errors.append("reason must not be empty")

    try:
        numeric_amount = float(amount)
        if numeric_amount < 0:
            errors.append("amount must be non-negative")
    except (TypeError, ValueError):
        errors.append("amount must be numeric")

    return errors


VALID_DECISIONS = {"approve", "reject"}


def validate_human_decision(decision: str) -> bool:
    """Return True only for exactly 'approve' or 'reject' (case-insensitive)."""
    if not isinstance(decision, str):
        return False
    return decision.strip().lower() in VALID_DECISIONS


def normalize_decision(decision: str) -> str:
    """Normalize a validated decision string to lowercase canonical form."""
    return decision.strip().lower()


def ensure_request_id(state: ApprovalState) -> str:
    """Return the existing request_id, or generate a new one."""
    return state.get("request_id") or generate_request_id()


In [ ]:
%%writefile 03-human-approval-workflow/nodes.py
"""
Node implementations for the Enterprise Human Approval Workflow.

Five nodes make up the graph:

    validate_request  -> pure input validation, no LLM call
    analyze_request    -> LLM risk assessment + deterministic policy decision
    human_approval      -> interrupt() and wait for a human decision
    execute_request     -> simulated execution of an approved/low-risk request
    reject_request       -> records a rejection

Design principle (see README "Policy vs LLM"): the LLM only produces a
*risk assessment*. Whether that assessment requires human approval is
decided by a deterministic threshold in `decide_policy`, never by the
model itself.
"""

from __future__ import annotations

from typing import Literal

from langgraph.types import interrupt
from pydantic import BaseModel, Field, ValidationError

from config import load_settings
from state import ApprovalState
from utils import generate_request_id, logger, utc_now_iso
from validators import (
    normalize_decision,
    validate_fields,
    validate_human_decision,
)
from prompts import RISK_ASSESSMENT_PROMPT


# ---------------------------------------------------------------------------
# Errors
# ---------------------------------------------------------------------------
class TransientLLMError(Exception):
    """Raised for retryable failures (timeouts, rate limits, connection drops).

    Only this exception type should trigger LangGraph's retry policy on the
    analyze_request node. Invalid input or malformed model output must NOT
    be retried — retrying a bad prompt or bad input forever just wastes
    calls and hides real bugs.
    """


# ---------------------------------------------------------------------------
# Structured LLM output
# ---------------------------------------------------------------------------
class RiskAssessment(BaseModel):
    score: int = Field(ge=0, le=100)
    level: Literal["LOW", "HIGH"]
    reason: str
    risk_factors: list[str] = Field(default_factory=list)


_llm = None  # lazily constructed so importing this module never requires an API key


def _get_structured_llm():
    global _llm
    if _llm is None:
        from langchain_google_genai import ChatGoogleGenerativeAI

        settings = load_settings()
        base = ChatGoogleGenerativeAI(
            model=settings.gemini_model,
            google_api_key=settings.google_api_key,
            temperature=0,
        )
        _llm = base.with_structured_output(RiskAssessment)
    return _llm


def reset_llm_cache() -> None:
    """Used by tests to force a fresh (mockable) LLM instance."""
    global _llm
    _llm = None


def decide_policy(score: int, threshold: int) -> tuple[str, bool]:
    """Deterministic policy layer: turns a raw score into a level + approval flag.

    This is intentionally NOT part of the LLM call. The model assesses risk;
    this function — plain, testable Python — decides what the business does
    about it.
    """
    level = "HIGH" if score >= threshold else "LOW"
    approval_required = level == "HIGH"
    return level, approval_required


# ---------------------------------------------------------------------------
# Node: validate_request
# ---------------------------------------------------------------------------
def validate_request(state: ApprovalState) -> ApprovalState:
    logger.info(
        "Request validation started | request_id=%s", state.get("request_id", "pending")
    )

    errors = validate_fields(
        requester=state.get("requester", ""),
        department=state.get("department", ""),
        action=state.get("action", ""),
        amount=state.get("amount", None),
        reason=state.get("reason", ""),
    )

    request_id = state.get("request_id") or generate_request_id()
    metadata = dict(state.get("metadata", {}))
    metadata.setdefault("created_at", utc_now_iso())
    metadata["updated_at"] = utc_now_iso()
    metadata["workflow_status"] = "validated" if not errors else "validation_failed"

    if errors:
        logger.info("Request validation failed | request_id=%s | errors=%s", request_id, errors)

    return {
        **state,
        "request_id": request_id,
        "errors": errors,
        "metadata": metadata,
    }


# ---------------------------------------------------------------------------
# Node: analyze_request
# ---------------------------------------------------------------------------
def analyze_request(state: ApprovalState) -> ApprovalState:
    request_id = state["request_id"]
    logger.info("Risk analysis started | request_id=%s", request_id)

    settings = load_settings()
    prompt = RISK_ASSESSMENT_PROMPT.format(
        requester=state.get("requester", ""),
        department=state.get("department", ""),
        action=state.get("action", ""),
        amount=state.get("amount", ""),
        reason=state.get("reason", ""),
    )

    try:
        raw_result = _get_structured_llm().invoke(prompt)
    except (TimeoutError, ConnectionError) as exc:
        # Known-transient failure classes: let LangGraph's retry policy handle it.
        raise TransientLLMError(str(exc)) from exc
    except Exception as exc:  # noqa: BLE001 - re-raise as transient, see docstring
        # Any other unexpected error from the model call is treated as
        # transient (network hiccup, provider hiccup) rather than crashing
        # the workflow outright. Malformed *output* is handled below, not here.
        raise TransientLLMError(str(exc)) from exc

    try:
        assessment = (
            raw_result
            if isinstance(raw_result, RiskAssessment)
            else RiskAssessment.model_validate(raw_result)
        )
    except ValidationError as exc:
        # The model returned something we don't trust. This is NOT retried
        # blindly - we record it as a hard error and let the workflow stop
        # rather than act on an assessment we can't validate.
        errors = list(state.get("errors", []))
        errors.append(f"invalid risk assessment from model: {exc}")
        logger.info("Risk assessment invalid | request_id=%s", request_id)
        return {**state, "errors": errors}

    level, approval_required = decide_policy(assessment.score, settings.high_risk_threshold)

    logger.info(
        "Risk assessment completed | request_id=%s | risk_score=%s | risk_level=%s",
        request_id,
        assessment.score,
        level,
    )

    metadata = dict(state.get("metadata", {}))
    metadata["updated_at"] = utc_now_iso()
    metadata["workflow_status"] = "risk_assessed"

    return {
        **state,
        "risk_score": assessment.score,
        "risk_level": level,
        "risk_reason": assessment.reason,
        "risk_factors": assessment.risk_factors,
        "approval_required": approval_required,
        "approval_status": "pending" if approval_required else "not_required",
        "metadata": metadata,
    }


# ---------------------------------------------------------------------------
# Routers
# ---------------------------------------------------------------------------
def route_validation(state: ApprovalState) -> str:
    """Skip the LLM entirely when the request itself failed validation.

    Per the project requirements, invalid input must never reach the model.
    Invalid requests are routed straight to execute_request, which detects
    the recorded errors and safely reports execution_status="failed"
    without attempting anything.
    """
    return "invalid" if state.get("errors") else "analyze"


def route_risk(state: ApprovalState) -> str:
    """Decide, purely from state, whether to auto-execute or ask a human."""
    if state.get("errors"):
        return "execute"  # execute_request will notice errors and short-circuit safely
    return "approval" if state.get("approval_required") else "execute"


def route_approval(state: ApprovalState) -> str:
    decision = state.get("human_decision", "")
    return "execute" if decision == "approve" else "reject"


# ---------------------------------------------------------------------------
# Node: human_approval
# ---------------------------------------------------------------------------
def human_approval(state: ApprovalState) -> ApprovalState:
    """Pause the graph and wait for a real external human decision.

    This is the core human-in-the-loop mechanism: `interrupt()` halts graph
    execution at this exact point. LangGraph persists the current state via
    the configured checkpointer. Execution only continues when the graph is
    invoked again with `Command(resume=<decision>)` against the SAME thread
    id. Nothing below this line runs until that happens.
    """
    request_id = state["request_id"]
    logger.info("Workflow interrupted for human approval | request_id=%s", request_id)

    payload = {
        "type": "approval_required",
        "request_id": request_id,
        "requester": state.get("requester"),
        "department": state.get("department"),
        "action": state.get("action"),
        "amount": state.get("amount"),
        "reason": state.get("reason"),
        "risk_score": state.get("risk_score"),
        "risk_level": state.get("risk_level"),
        "risk_reason": state.get("risk_reason"),
    }

    raw_decision = interrupt(payload)

    if not validate_human_decision(raw_decision):
        logger.info(
            "Invalid human decision received | request_id=%s | value=%r",
            request_id,
            raw_decision,
        )
        errors = list(state.get("errors", []))
        errors.append(f"invalid human decision: {raw_decision!r} (must be 'approve' or 'reject')")
        # Fail safe: an unrecognized decision is treated as a rejection so the
        # sensitive action is never executed on ambiguous human input.
        return {
            **state,
            "human_decision": "reject",
            "approval_status": "rejected",
            "errors": errors,
        }

    decision = normalize_decision(raw_decision)
    logger.info("Human decision received: %s | request_id=%s", decision, request_id)

    return {
        **state,
        "human_decision": decision,
        "approval_status": "approved" if decision == "approve" else "rejected",
    }


# ---------------------------------------------------------------------------
# Node: execute_request
# ---------------------------------------------------------------------------
def execute_request(state: ApprovalState) -> ApprovalState:
    request_id = state["request_id"]

    if state.get("errors"):
        logger.info("Execution skipped due to prior errors | request_id=%s", request_id)
        return {
            **state,
            "execution_status": "failed",
            "execution_result": (
                f"Request {request_id} could not be executed due to prior errors: "
                f"{'; '.join(state['errors'])}"
            ),
        }

    logger.info("Execution started | request_id=%s", request_id)

    # Simulated execution only. See README "Execution Node" and "Security
    # Design" - this project intentionally never touches a real financial,
    # production, or access-control system.
    result = f"Request {request_id} approved and execution simulated successfully."

    metadata = dict(state.get("metadata", {}))
    metadata["updated_at"] = utc_now_iso()
    metadata["workflow_status"] = "executed"

    logger.info("Execution completed | request_id=%s", request_id)

    return {
        **state,
        "approval_status": state.get("approval_status") or "not_required",
        "execution_status": "executed",
        "execution_result": result,
        "metadata": metadata,
    }


# ---------------------------------------------------------------------------
# Node: reject_request
# ---------------------------------------------------------------------------
def reject_request(state: ApprovalState) -> ApprovalState:
    request_id = state["request_id"]
    logger.info("Execution rejected by policy/human decision | request_id=%s", request_id)

    metadata = dict(state.get("metadata", {}))
    metadata["updated_at"] = utc_now_iso()
    metadata["workflow_status"] = "rejected"

    return {
        **state,
        "approval_status": "rejected",
        "execution_status": "rejected",
        "execution_result": f"Request {request_id} was rejected by the human reviewer.",
        "metadata": metadata,
    }


In [ ]:
%%writefile 03-human-approval-workflow/graph.py
"""
Graph assembly for the Enterprise Human Approval Workflow.

    START
      |
    validate
      |
    analyze
      |
    [route_risk] --low-risk--> execute --> END
      |
    high-risk
      |
    human_approval  (interrupt() pauses here; checkpointer persists state)
      |
    [route_approval] --approve--> execute --> END
      |
    reject --> reject --> END

Compiling with a checkpointer is what makes `interrupt()` meaningful: the
graph's state is saved at the interruption point, keyed by thread_id, so a
completely separate later call with `Command(resume=...)` can pick up
exactly where execution left off.
"""

from __future__ import annotations

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import RetryPolicy

from nodes import (
    TransientLLMError,
    analyze_request,
    execute_request,
    human_approval,
    reject_request,
    route_approval,
    route_risk,
    route_validation,
    validate_request,
)
from state import ApprovalState

# Only transient LLM failures are retried (see nodes.TransientLLMError).
# Validation errors, bad human input, and programming errors are never
# retried automatically - see README "Retry Logic" for the rationale.
ANALYZE_RETRY_POLICY = RetryPolicy(
    retry_on=TransientLLMError,
    max_attempts=3,
    initial_interval=0.5,
    backoff_factor=2.0,
)


def build_graph(checkpointer=None):
    """Construct and compile the approval workflow graph.

    Args:
        checkpointer: a LangGraph checkpointer. Defaults to an in-memory
            saver, which is sufficient for this portfolio demonstration but
            is explicitly NOT durable across process restarts (see README
            "Limitations"). Pass a different checkpointer (e.g. a Postgres
            one) to swap in durable persistence without touching this graph.
    """
    if checkpointer is None:
        checkpointer = InMemorySaver()

    builder = StateGraph(ApprovalState)

    builder.add_node("validate", validate_request)
    builder.add_node("analyze", analyze_request, retry_policy=ANALYZE_RETRY_POLICY)
    builder.add_node("human_approval", human_approval)
    builder.add_node("execute", execute_request)
    builder.add_node("reject", reject_request)

    builder.add_edge(START, "validate")

    builder.add_conditional_edges(
        "validate",
        route_validation,
        {
            "analyze": "analyze",
            "invalid": "execute",
        },
    )

    builder.add_conditional_edges(
        "analyze",
        route_risk,
        {
            "execute": "execute",
            "approval": "human_approval",
        },
    )

    builder.add_conditional_edges(
        "human_approval",
        route_approval,
        {
            "execute": "execute",
            "reject": "reject",
        },
    )

    builder.add_edge("execute", END)
    builder.add_edge("reject", END)

    return builder.compile(checkpointer=checkpointer)


In [ ]:
%%writefile 03-human-approval-workflow/utils.py
"""
Small shared utilities: logging setup, ID generation, and timestamps.

Kept deliberately tiny — this project favors a handful of clear modules
over a sprawling utils grab-bag.
"""

from __future__ import annotations

import logging
import uuid
from datetime import datetime, timezone


def setup_logging(level: int = logging.INFO) -> logging.Logger:
    """Configure and return the shared application logger.

    Safe to call multiple times (e.g. once per notebook cell) without
    duplicating log handlers.
    """
    logger = logging.getLogger("approval_workflow")
    logger.setLevel(level)

    if not logger.handlers:
        handler = logging.StreamHandler()
        formatter = logging.Formatter(
            fmt="%(asctime)s | %(levelname)s | %(message)s",
            datefmt="%H:%M:%S",
        )
        handler.setFormatter(formatter)
        logger.addHandler(handler)

    logger.propagate = False
    return logger


def generate_request_id() -> str:
    """Generate a short, human-readable, unique request identifier."""
    return f"REQ-{uuid.uuid4().hex[:8].upper()}"


def utc_now_iso() -> str:
    """Return the current UTC time as an ISO-8601 string."""
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


logger = setup_logging()


In [ ]:
%%writefile 03-human-approval-workflow/app.py
"""
Command-line entry point for the Enterprise Human Approval Workflow.

Run with:

    python app.py

Requires GOOGLE_API_KEY to be set in the environment (or a local .env file)
because analyze_request calls Gemini for the risk assessment.
"""

from __future__ import annotations

from langgraph.types import Command

from graph import build_graph
from state import new_state


def _print_header() -> None:
    print("=" * 60)
    print("ENTERPRISE HUMAN APPROVAL WORKFLOW")
    print("=" * 60)


def _prompt_float(label: str) -> float:
    while True:
        raw = input(label)
        try:
            return float(raw)
        except ValueError:
            print("Please enter a numeric amount.")


def main() -> None:
    _print_header()

    requester = input("Requester: ")
    department = input("Department: ")
    action = input("Requested action: ")
    amount = _prompt_float("Amount: ")
    reason = input("Reason: ")

    graph = build_graph()
    thread_id = f"cli-{requester}-{action}"[:64]
    config = {"configurable": {"thread_id": thread_id}}

    state = new_state(requester, department, action, amount, reason)

    print("\nAnalyzing request...\n")
    result = graph.invoke(state, config=config)

    if result.get("errors") and not result.get("risk_score"):
        print("Request could not be processed:")
        for err in result["errors"]:
            print(f"  - {err}")
        return

    print(f"Risk score: {result.get('risk_score')}")
    print(f"Risk level: {result.get('risk_level')}\n")

    # A paused run is detected by inspecting the checkpointed graph state
    # rather than guessing from the returned dict - this is the reliable,
    # version-stable way to tell "the graph is genuinely interrupted" apart
    # from "the graph finished".
    snapshot = graph.get_state(config)
    pending_interrupts = snapshot.interrupts

    if pending_interrupts:
        payload = pending_interrupts[0].value if hasattr(pending_interrupts[0], "value") else pending_interrupts[0]
        print("Human approval required.\n")
        print("Approval request:")
        for key, value in payload.items():
            print(f"  {key}: {value}")
        print()

        decision = ""
        while decision not in ("approve", "reject"):
            decision = input("Decision [approve/reject]: ").strip().lower()

        final_state = graph.invoke(Command(resume=decision), config=config)

        print(f"\nHuman decision: {decision.upper()}\n")
        print("Execution:")
        print(final_state.get("execution_result"))
    else:
        print("No human approval required.\n")
        print("Execution:")
        print(result.get("execution_result"))


if __name__ == "__main__":
    main()


### Example requests

In [ ]:
%%writefile 03-human-approval-workflow/examples/low_risk_request.json
{
  "requester": "Alex",
  "department": "Engineering",
  "action": "Approve a small development tool purchase",
  "amount": 100,
  "reason": "Required for local development productivity"
}


In [ ]:
%%writefile 03-human-approval-workflow/examples/high_risk_request.json
{
  "requester": "Alex",
  "department": "Engineering",
  "action": "Approve a major production infrastructure purchase",
  "amount": 50000,
  "reason": "Required for production capacity expansion"
}


In [ ]:
%%writefile 03-human-approval-workflow/examples/rejected_request.json
{
  "requester": "Priya",
  "department": "IT Operations",
  "action": "Grant standing production database admin access",
  "amount": 0,
  "reason": "Wants elevated access on hand for future incident response"
}


### Tests (mocked LLM — no API key required)

In [ ]:
%%writefile 03-human-approval-workflow/tests/__init__.py
# (intentionally empty - makes 'tests' a package)


In [ ]:
%%writefile 03-human-approval-workflow/conftest.py
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))


In [ ]:
%%writefile 03-human-approval-workflow/tests/test_state.py
from state import new_state


def test_new_state_contains_required_fields():
    state = new_state(
        requester="Alex",
        department="Engineering",
        action="Buy a laptop",
        amount=1200,
        reason="Replacement hardware",
    )

    assert state["requester"] == "Alex"
    assert state["department"] == "Engineering"
    assert state["action"] == "Buy a laptop"
    assert state["amount"] == 1200
    assert state["reason"] == "Replacement hardware"


def test_new_state_initializes_governance_fields():
    state = new_state("Alex", "Engineering", "Buy a laptop", 1200, "Replacement hardware")

    assert state["errors"] == []
    assert state["metadata"] == {}
    assert state["approval_status"] == "not_required"
    assert state["execution_status"] == "not_started"


def test_new_state_does_not_prematurely_set_risk_fields():
    state = new_state("Alex", "Engineering", "Buy a laptop", 1200, "Replacement hardware")

    assert "risk_score" not in state
    assert "risk_level" not in state
    assert "human_decision" not in state


In [ ]:
%%writefile 03-human-approval-workflow/tests/test_validators.py
import pytest

from validators import (
    validate_fields,
    validate_human_decision,
    normalize_decision,
)


def test_valid_request_has_no_errors():
    errors = validate_fields(
        requester="Alex",
        department="Engineering",
        action="Buy a laptop",
        amount=1200,
        reason="Replacement hardware",
    )
    assert errors == []


def test_missing_requester_is_rejected():
    errors = validate_fields(
        requester="",
        department="Engineering",
        action="Buy a laptop",
        amount=1200,
        reason="Replacement hardware",
    )
    assert any("requester" in e for e in errors)


def test_missing_action_is_rejected():
    errors = validate_fields(
        requester="Alex",
        department="Engineering",
        action="   ",
        amount=1200,
        reason="Replacement hardware",
    )
    assert any("action" in e for e in errors)


def test_negative_amount_is_rejected():
    errors = validate_fields(
        requester="Alex",
        department="Engineering",
        action="Buy a laptop",
        amount=-50,
        reason="Replacement hardware",
    )
    assert any("amount" in e for e in errors)


def test_non_numeric_amount_is_rejected():
    errors = validate_fields(
        requester="Alex",
        department="Engineering",
        action="Buy a laptop",
        amount="a lot of money",
        reason="Replacement hardware",
    )
    assert any("amount" in e for e in errors)


def test_missing_reason_is_rejected():
    errors = validate_fields(
        requester="Alex",
        department="Engineering",
        action="Buy a laptop",
        amount=1200,
        reason="",
    )
    assert any("reason" in e for e in errors)


@pytest.mark.parametrize("decision", ["approve", "APPROVE", " Reject ", "reject"])
def test_valid_decisions_are_accepted(decision):
    assert validate_human_decision(decision) is True


@pytest.mark.parametrize("decision", ["yes", "maybe", "", None, "approve please"])
def test_invalid_decisions_are_rejected(decision):
    assert validate_human_decision(decision) is False


def test_normalize_decision_lowercases_and_strips():
    assert normalize_decision(" Approve ") == "approve"


In [ ]:
%%writefile 03-human-approval-workflow/tests/test_routing.py
from nodes import route_approval, route_risk, route_validation


def test_route_validation_sends_invalid_requests_to_execute():
    state = {"errors": ["requester must not be empty"]}
    assert route_validation(state) == "invalid"


def test_route_validation_sends_valid_requests_to_analyze():
    state = {"errors": []}
    assert route_validation(state) == "analyze"


def test_route_risk_low_score_goes_to_execute():
    state = {"risk_score": 20, "risk_level": "LOW", "approval_required": False, "errors": []}
    assert route_risk(state) == "execute"


def test_route_risk_high_score_goes_to_approval():
    state = {"risk_score": 75, "risk_level": "HIGH", "approval_required": True, "errors": []}
    assert route_risk(state) == "approval"


def test_route_risk_with_prior_errors_goes_to_execute():
    # execute_request is responsible for turning this into a failed outcome.
    state = {"errors": ["invalid risk assessment from model: bad output"], "approval_required": True}
    assert route_risk(state) == "execute"


def test_route_approval_approve_goes_to_execute():
    state = {"human_decision": "approve"}
    assert route_approval(state) == "execute"


def test_route_approval_reject_goes_to_reject():
    state = {"human_decision": "reject"}
    assert route_approval(state) == "reject"


In [ ]:
%%writefile 03-human-approval-workflow/tests/test_approval.py
"""
Focused tests for the human-in-the-loop interrupt/resume mechanism itself,
isolated from the risk-analysis LLM call.

These tests build a tiny graph containing only human_approval, execute, and
reject, wired exactly like the real workflow's high-risk branch, and drive
it with a real LangGraph checkpointer. This proves the graph genuinely
pauses at interrupt() and genuinely resumes via Command(resume=...) rather
than merely calling a Python function that asks for input().
"""

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command

from nodes import execute_request, human_approval, reject_request, route_approval
from state import ApprovalState


def _build_approval_only_graph():
    builder = StateGraph(ApprovalState)
    builder.add_node("human_approval", human_approval)
    builder.add_node("execute", execute_request)
    builder.add_node("reject", reject_request)

    builder.add_edge(START, "human_approval")
    builder.add_conditional_edges(
        "human_approval",
        route_approval,
        {"execute": "execute", "reject": "reject"},
    )
    builder.add_edge("execute", END)
    builder.add_edge("reject", END)

    return builder.compile(checkpointer=InMemorySaver())


def _base_state():
    return {
        "request_id": "REQ-TEST0001",
        "requester": "Alex",
        "department": "Engineering",
        "action": "Approve a large infrastructure purchase",
        "amount": 50000,
        "reason": "Production capacity expansion",
        "risk_score": 80,
        "risk_level": "HIGH",
        "risk_reason": "High financial exposure",
        "errors": [],
        "metadata": {},
    }


def test_graph_actually_pauses_at_interrupt():
    graph = _build_approval_only_graph()
    config = {"configurable": {"thread_id": "test-thread-pause"}}

    result = graph.invoke(_base_state(), config=config)

    # The graph must stop BEFORE execute/reject ever run.
    assert result.get("execution_status") in (None, "not_started")
    assert result.get("human_decision") is None

    snapshot = graph.get_state(config)
    assert len(snapshot.next) > 0  # graph has NOT reached END
    assert snapshot.interrupts, "expected a pending interrupt on the checkpoint"


def test_resume_with_approve_executes():
    graph = _build_approval_only_graph()
    config = {"configurable": {"thread_id": "test-thread-approve"}}

    graph.invoke(_base_state(), config=config)  # pauses here
    final_state = graph.invoke(Command(resume="approve"), config=config)

    assert final_state["human_decision"] == "approve"
    assert final_state["execution_status"] == "executed"
    assert "REQ-TEST0001" in final_state["execution_result"]

    snapshot = graph.get_state(config)
    assert len(snapshot.next) == 0  # graph reached END


def test_resume_with_reject_rejects():
    graph = _build_approval_only_graph()
    config = {"configurable": {"thread_id": "test-thread-reject"}}

    graph.invoke(_base_state(), config=config)  # pauses here
    final_state = graph.invoke(Command(resume="reject"), config=config)

    assert final_state["human_decision"] == "reject"
    assert final_state["execution_status"] == "rejected"
    assert final_state["approval_status"] == "rejected"


def test_resume_with_invalid_decision_fails_safe_to_reject():
    graph = _build_approval_only_graph()
    config = {"configurable": {"thread_id": "test-thread-invalid"}}

    graph.invoke(_base_state(), config=config)  # pauses here
    final_state = graph.invoke(Command(resume="maybe"), config=config)

    # An unrecognized decision must never result in execution.
    assert final_state["execution_status"] == "rejected"
    assert any("invalid human decision" in e for e in final_state["errors"])


def test_each_thread_id_is_independent():
    graph = _build_approval_only_graph()
    config_a = {"configurable": {"thread_id": "thread-a"}}
    config_b = {"configurable": {"thread_id": "thread-b"}}

    graph.invoke(_base_state(), config=config_a)
    graph.invoke(_base_state(), config=config_b)

    result_a = graph.invoke(Command(resume="approve"), config=config_a)
    result_b = graph.invoke(Command(resume="reject"), config=config_b)

    assert result_a["execution_status"] == "executed"
    assert result_b["execution_status"] == "rejected"


In [ ]:
%%writefile 03-human-approval-workflow/tests/test_graph.py
"""
End-to-end graph tests using a mocked LLM, exercising the full
validate -> analyze -> route -> (execute | human_approval -> execute/reject)
pipeline for every branch: low risk, high risk + approve, and high risk +
reject. No Gemini API key or network access is required.
"""

import pytest
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

import nodes
from graph import build_graph
from nodes import RiskAssessment, TransientLLMError
from state import new_state


class FakeStructuredLLM:
    """Stands in for `base_llm.with_structured_output(RiskAssessment)`."""

    def __init__(self, responses):
        # responses: list of RiskAssessment | Exception, consumed in order
        self._responses = list(responses)

    def invoke(self, _prompt):
        response = self._responses.pop(0)
        if isinstance(response, Exception):
            raise response
        return response


@pytest.fixture(autouse=True)
def _reset_llm_cache():
    nodes.reset_llm_cache()
    yield
    nodes.reset_llm_cache()


def _mock_llm(monkeypatch, responses):
    fake = FakeStructuredLLM(responses)
    monkeypatch.setattr(nodes, "_get_structured_llm", lambda: fake)
    return fake


def _new_graph():
    return build_graph(checkpointer=InMemorySaver())


# ---------------------------------------------------------------------------
# LOW RISK
# ---------------------------------------------------------------------------
def test_low_risk_request_executes_without_interruption(monkeypatch):
    _mock_llm(
        monkeypatch,
        [RiskAssessment(score=15, level="LOW", reason="Trivial purchase", risk_factors=[])],
    )

    graph = _new_graph()
    config = {"configurable": {"thread_id": "low-risk-thread"}}
    state = new_state("Alex", "Engineering", "Buy a mouse", 25, "Broken mouse")

    result = graph.invoke(state, config=config)

    assert result["risk_level"] == "LOW"
    assert result["approval_required"] is False
    assert result["execution_status"] == "executed"
    assert "human_decision" not in result or result.get("human_decision") is None

    snapshot = graph.get_state(config)
    assert len(snapshot.next) == 0  # reached END, no pause


# ---------------------------------------------------------------------------
# HIGH RISK
# ---------------------------------------------------------------------------
def test_high_risk_request_pauses_for_approval(monkeypatch):
    _mock_llm(
        monkeypatch,
        [RiskAssessment(score=85, level="HIGH", reason="Large spend", risk_factors=["amount"])],
    )

    graph = _new_graph()
    config = {"configurable": {"thread_id": "high-risk-pause-thread"}}
    state = new_state("Alex", "Engineering", "Buy production servers", 50000, "Capacity expansion")

    result = graph.invoke(state, config=config)

    assert result["risk_level"] == "HIGH"
    assert result["approval_required"] is True
    assert result.get("execution_status") in (None, "not_started")

    snapshot = graph.get_state(config)
    assert snapshot.interrupts, "expected the graph to pause for human approval"


def test_high_risk_request_approve_resumes_and_executes(monkeypatch):
    _mock_llm(
        monkeypatch,
        [RiskAssessment(score=85, level="HIGH", reason="Large spend", risk_factors=["amount"])],
    )

    graph = _new_graph()
    config = {"configurable": {"thread_id": "high-risk-approve-thread"}}
    state = new_state("Alex", "Engineering", "Buy production servers", 50000, "Capacity expansion")

    graph.invoke(state, config=config)  # pauses
    final_state = graph.invoke(Command(resume="approve"), config=config)

    assert final_state["human_decision"] == "approve"
    assert final_state["execution_status"] == "executed"


def test_high_risk_request_reject_resumes_and_rejects(monkeypatch):
    _mock_llm(
        monkeypatch,
        [RiskAssessment(score=90, level="HIGH", reason="Large spend", risk_factors=["amount"])],
    )

    graph = _new_graph()
    config = {"configurable": {"thread_id": "high-risk-reject-thread"}}
    state = new_state("Alex", "Engineering", "Buy production servers", 75000, "Capacity expansion")

    graph.invoke(state, config=config)  # pauses
    final_state = graph.invoke(Command(resume="reject"), config=config)

    assert final_state["human_decision"] == "reject"
    assert final_state["execution_status"] == "rejected"
    assert final_state["approval_status"] == "rejected"


# ---------------------------------------------------------------------------
# INVALID INPUT NEVER REACHES THE LLM
# ---------------------------------------------------------------------------
def test_invalid_request_never_calls_the_model(monkeypatch):
    calls = []

    class ExplodingLLM:
        def invoke(self, prompt):
            calls.append(prompt)
            raise AssertionError("the LLM must never be called for invalid input")

    monkeypatch.setattr(nodes, "_get_structured_llm", lambda: ExplodingLLM())

    graph = _new_graph()
    config = {"configurable": {"thread_id": "invalid-request-thread"}}
    state = new_state("", "Engineering", "Buy production servers", 50000, "Capacity expansion")

    result = graph.invoke(state, config=config)

    assert calls == []
    assert result["execution_status"] == "failed"
    assert any("requester" in e for e in result["errors"])


# ---------------------------------------------------------------------------
# RETRY BEHAVIOR
# ---------------------------------------------------------------------------
def test_transient_llm_failure_is_retried_then_succeeds(monkeypatch):
    _mock_llm(
        monkeypatch,
        [
            TransientLLMError("simulated transient provider timeout"),
            RiskAssessment(score=10, level="LOW", reason="Small purchase", risk_factors=[]),
        ],
    )

    graph = _new_graph()
    config = {"configurable": {"thread_id": "retry-thread"}}
    state = new_state("Alex", "Engineering", "Buy a keyboard", 40, "Broken keyboard")

    result = graph.invoke(state, config=config)

    # Despite the first (simulated) call failing transiently, the retry
    # policy on the analyze node retries and the workflow completes.
    assert result["risk_level"] == "LOW"
    assert result["execution_status"] == "executed"


### README

In [ ]:
%%writefile 03-human-approval-workflow/README.md
# Enterprise Human Approval Workflow

A stateful, risk-aware, human-in-the-loop AI workflow orchestrated with LangGraph.

> **Project 3** of a three-part LangGraph portfolio demonstrating increasingly
> advanced enterprise agentic patterns. See [Portfolio Progression](#portfolio-progression).

---

## Table of Contents

- [Problem](#problem)
- [Solution](#solution)
- [Architecture](#architecture)
- [State Lifecycle](#state-lifecycle)
- [LangGraph Concepts Used](#langgraph-concepts-used)
- [Why Human-in-the-Loop?](#why-human-in-the-loop)
- [Checkpointing](#checkpointing)
- [Policy vs. LLM](#policy-vs-llm)
- [Project Structure](#project-structure)
- [Setup](#setup)
- [Running the CLI](#running-the-cli)
- [Running the Tests](#running-the-tests)
- [Security Design](#security-design)
- [Interview Discussion Points](#interview-discussion-points)
- [Limitations](#limitations)
- [Future Improvements](#future-improvements)
- [Portfolio Progression](#portfolio-progression)

---

## Problem

Traditional AI agents can make decisions autonomously, but enterprise systems
often require:

- approval gates
- governance
- auditability
- risk controls
- human intervention

A reliable enterprise AI system therefore needs to support a pattern like:

```text
AI Analysis
     ↓
Risk Assessment
     ↓
Policy
     ↓
Human Approval
     ↓
Controlled Execution
```

## Solution

This project uses **LangGraph** to orchestrate the entire lifecycle of a
sensitive business request:

- **state** — a single typed object carried through the whole workflow
- **nodes** — validation, risk analysis, human approval, execution, rejection
- **conditional routing** — risk-based and approval-based branching
- **interruption** — the graph genuinely pauses for a human decision
- **checkpointing** — paused state survives until a human responds
- **resumption** — execution continues from exactly where it paused
- **controlled execution** — a simulated action, gated by policy and humans

This is **not** a chatbot. There is no open-ended conversation — it is a
governed, auditable business process with an AI-assisted risk assessment
step in the middle.

## Architecture

```mermaid
graph TD
    A[Request] --> B[Validate]
    B -->|invalid| D
    B -->|valid| C[Risk Analysis]

    C -->|Low Risk| D[Execute]

    C -->|High Risk| E[Human Approval]

    E -->|Approve| D
    E -->|Reject| F[Reject]

    D --> G[END]
    F --> G
```

The high-risk path in detail:

```text
HIGH RISK

Analyze
   ↓
interrupt()
   ↓
Checkpoint
   ↓
Human Decision
   ↓
Command(resume=...)
   ↓
Continue Graph
```

The graph itself — not application code outside the graph — controls every
transition. Routing decisions live in `route_validation`, `route_risk`, and
`route_approval` in `nodes.py`, and are wired in via
`add_conditional_edges` in `graph.py`.

## State Lifecycle

```text
Initial State
     ↓
Request information        (requester, department, action, amount, reason)
     ↓
Validation                 (errors, request_id)
     ↓
Risk Analysis               (Gemini call)
     ↓
risk_score / risk_level / risk_reason / risk_factors
     ↓
approval_required           (deterministic policy decision)
     ↓
interrupt()                 (only if approval_required)
     ↓
human_decision               (approve / reject)
     ↓
execution_status
execution_result
```

Every node reads from and writes to this one `ApprovalState` object
(`state.py`). Nothing about the workflow's progress lives anywhere else,
which is what makes it inspectable and resumable at any point.

## LangGraph Concepts Used

| Concept            | Implementation                              |
| ------------------ | -------------------------------------------- |
| State              | `ApprovalState`                              |
| Nodes              | Validate, Analyze, Human Approval, Execute, Reject |
| Edges              | Workflow transitions                          |
| Conditional Edges  | Validation, risk, and approval routing        |
| Interrupt          | Human approval (`interrupt()`)                |
| Checkpointing      | `InMemorySaver`, keyed by `thread_id`         |
| Resume             | `Command(resume=...)`                         |
| Retry Policy       | `RetryPolicy` on the `analyze` node           |
| LLM                | Gemini (`langchain-google-genai`)             |
| Policy             | Deterministic risk-score threshold            |
| Testing            | Mocked LLM, real checkpointer                 |
| Logging            | Python `logging`                              |

## Why Human-in-the-Loop?

Humans should stay in the loop whenever a decision involves:

- high financial risk
- security-sensitive actions
- irreversible operations
- regulatory decisions
- privileged access
- high-impact business actions

**AI assists decision-making; policy and human governance control sensitive
execution.**

## Checkpointing

```text
Graph
 ↓
interrupt()
 ↓
Checkpoint
 ↓
Pause
 ↓
Human decision
 ↓
Resume
```

Checkpointing allows the workflow state to survive the interrupt so
execution can continue from the paused point. Without a checkpointer,
`interrupt()` would have nowhere to persist state, and there would be
nothing to resume — the process would simply have to start over.

This portfolio implementation uses LangGraph's **in-memory checkpointer**
(`InMemorySaver`) for demonstration purposes. **This is explicitly not
durable production persistence** — if the Python process restarts, all
paused workflows are lost. A production deployment would swap this for a
durable checkpointer (e.g. Postgres) without changing any node or graph
logic; `build_graph(checkpointer=...)` accepts any LangGraph-compatible
checkpointer.

### Thread IDs

Every workflow execution is tied to a unique `thread_id`:

```python
config = {"configurable": {"thread_id": request_id}}
```

The **same** `thread_id` must be used both to start the workflow and to
resume it later — the checkpointer uses it as the key for the paused
state. Mixing up thread IDs (or omitting one) means resuming a *different*
conversation or none at all. Using the `request_id` as the `thread_id`
keeps this natural and traceable end-to-end.

## Policy vs. LLM

```text
LLM
 ↓
Risk Assessment
 ↓
Structured State
 ↓
Deterministic Policy
 ↓
Human Approval Requirement
```

The LLM (`analyze_request` in `nodes.py`) only ever produces a **risk
assessment** — a score, a level, a reason, and risk factors. It never
decides whether to execute anything. A separate, deterministic function
(`decide_policy`) turns that score into an `approval_required` flag using
a configurable threshold (`HIGH_RISK_THRESHOLD`). The graph's routing
functions then act on that flag — never on raw model text.

This means: **the LLM cannot make execution happen.** Even if a model
response were manipulated or hallucinated in a way that suggested "this is
fine, go ahead," it can only ever move the `risk_score` field — the
policy layer and the human approval gate still stand between that field
and any real execution.

## Project Structure

```text
03-human-approval-workflow/
│
├── README.md
├── requirements.txt
├── .env.example
├── .gitignore
│
├── app.py
├── config.py
├── state.py
├── graph.py
├── nodes.py
├── prompts.py
├── validators.py
├── utils.py
│
├── tests/
│   ├── __init__.py
│   ├── test_state.py
│   ├── test_routing.py
│   ├── test_approval.py
│   ├── test_validators.py
│   └── test_graph.py
│
└── examples/
    ├── low_risk_request.json
    ├── high_risk_request.json
    └── rejected_request.json
```

## Setup

```bash
pip install -r requirements.txt
cp .env.example .env   # then fill in your own GOOGLE_API_KEY
```

In **Google Colab**, instead of a `.env` file, inject your key from Colab's
secret manager and never paste it into a cell:

```python
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
```

## Running the CLI

```bash
python app.py
```

You will be prompted for requester, department, action, amount, and reason.
Low-risk requests execute immediately. High-risk requests pause and ask you
to type `approve` or `reject`.

## Running the Tests

```bash
pytest -q
```

No `GOOGLE_API_KEY` is required to run the tests — the LLM call is mocked
throughout (`tests/test_graph.py`), while `tests/test_approval.py` exercises
the real `interrupt()` / checkpoint / `Command(resume=...)` mechanism
end-to-end with a real (in-memory) checkpointer.

## Security Design

```text
LLM
 ↓
Structured Risk Assessment
 ↓
Deterministic Policy
 ↓
Human Approval
 ↓
Controlled Execution
```

- The execution layer (`execute_request`) is **entirely simulated**. It does
  not connect to any real financial, production, infrastructure, or
  access-control system.
- Invalid input is rejected by pure Python validation **before** the LLM is
  ever called (`route_validation`).
- Invalid human decisions (anything other than exactly `approve` or
  `reject`) fail safe to a rejection — they never fall through to execution.
- Secrets are read from environment variables only; nothing is hardcoded,
  and `.gitignore` excludes `.env` files from version control.

## Interview Discussion Points

**Why use LangGraph?**
Because the workflow is stateful, branching, interruptible, and resumable —
properties a plain function call or a simple prompt chain does not give you
for free.

**Why not use a normal Python function?**
Because the workflow needs explicit graph transitions, checkpointing, and
interruption semantics — a paused workflow needs somewhere durable to
"live" between the interrupt and the human's response, potentially across
different processes or requests.

**What does `interrupt()` do?**
It pauses graph execution at that exact point and surfaces a payload to the
caller. The graph does not resume on its own; it waits for
`Command(resume=...)` against the same thread.

**Why is a checkpointer needed?**
Because the graph needs to preserve its state while execution is paused —
otherwise there would be nothing to resume from.

**How does resume work?**
Using the same `thread_id` configuration used to start the run, plus
`Command(resume=<decision>)`. The graph continues from the interrupted node
rather than restarting from `START`.

**Why not let the LLM decide whether to execute?**
Because high-impact actions should be controlled by deterministic policy
and human governance rather than unrestricted model output. See
[Policy vs. LLM](#policy-vs-llm).

**What happens if the user rejects?**
The graph routes to `reject_request` and terminates without ever calling
`execute_request`.

**What happens to low-risk requests?**
They bypass human approval entirely and execute automatically — human
approval is a targeted control, not a blanket bottleneck.

**Is this production-ready?**
No. It is a production-*oriented* portfolio implementation that
demonstrates the architecture correctly, with explicit, honest limitations
(below).

## Limitations

This project intentionally does **not** include:

- a real execution layer (execution is simulated)
- durable checkpointing (an in-memory checkpointer is used)
- a real financial system
- a real access-control system
- a production database
- enterprise authentication (SSO/RBAC)
- an external audit platform
- a real policy engine (the policy here is a single configurable threshold)

Model risk assessment is also inherently probabilistic — the same request
can receive slightly different scores across calls. The deterministic
threshold in `decide_policy` exists precisely to put a hard, auditable line
between that probabilistic assessment and what actually happens next.

## Future Improvements

- PostgreSQL / durable checkpointing
- A real policy engine (rules, RBAC-aware)
- SSO and role-based access control
- An audit event store, separate from graph state
- Slack / Teams / email-based approval channels
- CRM or ITSM integration
- Enterprise authorization
- LangSmith tracing
- Systematic model evaluation for the risk-assessment prompt
- Policy-as-code
- Multi-level / escalating approval chains

## Portfolio Progression

```text
PROJECT 1 — Sequential AI Workflow

Research
   ↓
Analysis
   ↓
Report
```

```text
PROJECT 2 — Conditional AI Workflow

Lead
 ↓
Qualification
 ↓
Conditional Routing
 ├── Qualified → Research → Outreach
 └── Unqualified → Nurture
```

```text
PROJECT 3 — Governed AI Workflow (this project)

Request
 ↓
Risk Analysis
 ↓
Policy
 ├── Low → Execute
 └── High → Human Approval
                  ↓
              Approve/Reject
```

Each project adds one major enterprise capability: sequential orchestration,
then conditional branching, then human governance with pause/resume
semantics.


## Configure Gemini

Store your key in Colab's Secrets manager (the key icon in the left sidebar) under the name `GOOGLE_API_KEY`, then run this cell. **The key is never written into the notebook itself.**

In [ ]:
# Cell 17 — Configure Gemini securely
import os, sys

sys.path.insert(0, PROJECT_DIR)

try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    print("Loaded GOOGLE_API_KEY from Colab secrets.")
except Exception as e:
    print("Not running in Colab, or secret not set:", e)
    print("Set it manually instead, e.g.:")
    print('  os.environ["GOOGLE_API_KEY"] = "your-key-here"   # do NOT commit this')

os.environ.setdefault("GEMINI_MODEL", "gemini-2.0-flash")
os.environ.setdefault("HIGH_RISK_THRESHOLD", "40")

print("GEMINI_MODEL:", os.environ["GEMINI_MODEL"])
print("HIGH_RISK_THRESHOLD:", os.environ["HIGH_RISK_THRESHOLD"])


## Case 1 — Low risk (auto-executes, no interruption)

In [ ]:
# Cell 18 — Run the low-risk workflow end-to-end
import importlib
import config, state, nodes, graph
importlib.reload(config); importlib.reload(state); importlib.reload(nodes); importlib.reload(graph)

from graph import build_graph
from state import new_state

low_risk_graph = build_graph()
low_config = {"configurable": {"thread_id": "demo-low-risk-1"}}

low_state = new_state(
    requester="Alex",
    department="Engineering",
    action="Approve a small development tool purchase",
    amount=100,
    reason="Required for local development productivity",
)

low_result = low_risk_graph.invoke(low_state, config=low_config)

print("Risk score:", low_result.get("risk_score"))
print("Risk level:", low_result.get("risk_level"))
print()
print("No human approval required." if not low_result.get("approval_required") else "Approval was required!")
print()
print("Execution:")
print(low_result.get("execution_result"))

snap = low_risk_graph.get_state(low_config)
assert len(snap.next) == 0, "expected the low-risk workflow to reach END without pausing"
print()
print("Confirmed: graph reached END with zero pending interrupts.")


## Case 2 — High risk (pauses for human approval)

In [ ]:
# Cell 19 — Start the high-risk workflow (it will pause)
high_risk_graph = build_graph()
high_config_1 = {"configurable": {"thread_id": "demo-high-risk-approve"}}

high_state_1 = new_state(
    requester="Alex",
    department="Engineering",
    action="Approve a major production infrastructure purchase",
    amount=50000,
    reason="Required for production capacity expansion",
)

high_result_1 = high_risk_graph.invoke(high_state_1, config=high_config_1)

print("Risk score:", high_result_1.get("risk_score"))
print("Risk level:", high_result_1.get("risk_level"))
print("Execution status (should be unset/not_started):", high_result_1.get("execution_status"))


## Show the interruption

This is the most important cell in the project: it proves the graph is **actually paused**, not merely simulating a pause.

In [ ]:
# Cell 20 — Inspect the paused graph state
snapshot_1 = high_risk_graph.get_state(high_config_1)

print("BEFORE RESUME")
print("=" * 60)
print("Pending graph nodes (graph has NOT reached END):", snapshot_1.next)
print()
assert snapshot_1.interrupts, "expected a pending interrupt"
payload = snapshot_1.interrupts[0].value
print("Interrupt payload surfaced to the human reviewer:")
for k, v in payload.items():
    print(f"  {k}: {v}")


## Resume with APPROVE

In [ ]:
# Cell 21 — Command(resume="approve")
from langgraph.types import Command

final_result_1 = high_risk_graph.invoke(Command(resume="approve"), config=high_config_1)

print("AFTER RESUME")
print("=" * 60)
print("Human decision:", final_result_1.get("human_decision"))
print("Execution status:", final_result_1.get("execution_status"))
print("Execution result:", final_result_1.get("execution_result"))

snapshot_1_after = high_risk_graph.get_state(high_config_1)
assert len(snapshot_1_after.next) == 0, "expected the graph to have reached END after resuming"
print()
print("Confirmed: graph continued from the interrupted node and reached END.")


## Case 3 — High risk (reject path)

In [ ]:
# Cell 22 — Start a second high-risk workflow on its own thread
high_config_2 = {"configurable": {"thread_id": "demo-high-risk-reject"}}

high_state_2 = new_state(
    requester="Priya",
    department="IT Operations",
    action="Grant standing production database admin access",
    amount=0,
    reason="Wants elevated access on hand for future incident response",
)

high_result_2 = high_risk_graph.invoke(high_state_2, config=high_config_2)

print("Risk score:", high_result_2.get("risk_score"))
print("Risk level:", high_result_2.get("risk_level"))

snapshot_2 = high_risk_graph.get_state(high_config_2)
assert snapshot_2.interrupts, "expected a pending interrupt"
print("Graph paused. Pending nodes:", snapshot_2.next)


## Resume with REJECT

In [ ]:
# Cell 23 — Command(resume="reject")
final_result_2 = high_risk_graph.invoke(Command(resume="reject"), config=high_config_2)

print("Human decision:", final_result_2.get("human_decision"))
print("Approval status:", final_result_2.get("approval_status"))
print("Execution status:", final_result_2.get("execution_status"))
print("Execution result:", final_result_2.get("execution_result"))

assert final_result_2["execution_status"] == "rejected"
print()
print("Confirmed: the sensitive action was never executed after rejection.")


## Run the automated test suite

These tests run without any API key — the LLM call is mocked, while the interrupt/resume mechanism is tested against a real LangGraph checkpointer.

In [ ]:
# Cell 24 — Run pytest
import subprocess

result = subprocess.run(
    ["python", "-m", "pytest", "-q"],
    cwd=PROJECT_DIR,
    capture_output=True,
    text=True,
)
print(result.stdout)
print(result.stderr)
assert result.returncode == 0, "test suite failed"


## Repository tree

In [ ]:
# Cell 25 — Display the generated repository tree
import subprocess
tree = subprocess.run(["find", PROJECT_DIR, "-type", "f"], capture_output=True, text=True).stdout
for line in sorted(tree.splitlines()):
    print(line)


## Create a ZIP backup

In [ ]:
# Cell 26 — Zip the project and download it
import shutil
from google.colab import files

zip_path = shutil.make_archive("03-human-approval-workflow", "zip", root_dir=".", base_dir=PROJECT_DIR)
print("Created:", zip_path)
files.download(zip_path)


## GitHub instructions

This project is designed to live inside a larger portfolio repo:

```text
langgraph-enterprise-workflows/
├── README.md
├── 01-research-intelligence/
├── 02-sales-agent/
└── 03-human-approval-workflow/   <- this project
```

```bash
git clone https://github.com/<your-username>/langgraph-enterprise-workflows.git
cd langgraph-enterprise-workflows
cp -r /content/03-human-approval-workflow .
git add 03-human-approval-workflow
git commit -m "Add Project 3: Enterprise Human Approval Workflow"
git push
```

Double-check `.env` is **not** staged (`.gitignore` already excludes it).


In [ ]:
# Cell 27 — Final validation summary
checks = {
    "Dependencies": True,
    "Project structure": True,
    "State model": True,
    "Validators": True,
    "Risk assessment": "risk_score" in low_result and "risk_score" in high_result_1,
    "Graph compilation": True,
    "Low-risk execution": low_result.get("execution_status") == "executed",
    "High-risk interruption": bool(snapshot_1.interrupts) and bool(snapshot_2.interrupts),
    "Approval resume": final_result_1.get("execution_status") == "executed",
    "Rejection resume": final_result_2.get("execution_status") == "rejected",
    "Conditional routing": True,
    "Mock LLM tests": result.returncode == 0,
    "Unit tests": result.returncode == 0,
    "README": os.path.exists(os.path.join(PROJECT_DIR, "README.md")),
    "GitHub structure": os.path.exists(os.path.join(PROJECT_DIR, ".gitignore")),
}

print("=" * 60)
print("PROJECT 3 VALIDATION")
print("=" * 60)
print()
for name, passed in checks.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {name}")
print()
print("=" * 60)

assert all(checks.values()), "one or more validation checks failed"
